## 第14章　计算图与自动微分 —— 反向传播的工程本质> 本章目标：理解计算图（节点=操作，边=数据流）如何将链式法则自动化。手算一个简单计算图的反向传播，然后用 PyTorch 的 `.backward()` 验证。掌握 `detach()` 和 `torch.no_grad()` 两个梯度控制工具——前者断开单个张量，后者关闭整个代码块的梯度追踪。> 前置知识：第 13 章（链式法则）、第 6 章（矩阵乘法）、第 12 章（梯度）---### 14.1　把表达式画成节点图 —— 计算图链式法则在纸上算很简单，但让计算机自动对任意表达式执行链式法则，需要一个数据结构来记录"谁依赖谁"——这就是**计算图（Computational Graph）**。每个节点是一个操作（+、×、sin、matmul...），边是数据流动的方向。以 `y = (x+2)×3` 为例：```x ──→ [+2] ──→ u ──→ [×3] ──→ y```📐 **定义　计算图（Computational Graph）**：有向无环图，节点=数学操作，边=张量。前向沿边计算结果，反向沿边传递梯度。---### 14.2　前向传播 —— 从 x 走到 y前向传播就是沿着计算图从左到右依次计算。对于 `y = (x+2)×3`：x=1 → u=x+2=3 → y=u×3=9。每一步只做一件事——加或乘——但串起来就完成了从输入到输出的完整计算。---### 14.3　反向传播（手算）—— 从输出往回走反向传播沿计算图从右到左传递梯度。**每个操作节点只需要知道两件事**：(1) 输出对输入的**本地梯度**（如乘法节点 ∂(3u)/∂u=3），(2) 从上游传来的**累积梯度**。两者相乘，传给下游。对于 `y = (x+2)×3`：∂y/∂y=1（起点）→ ∂y/∂u=3（乘节点本地梯度）→ 累积 3×1=3 → ∂u/∂x=1（加节点本地梯度）→ 最终 ∂y/∂x = 3×1 = 3。📐 **反向传播 = 链式法则在计算图上的逐节点执行。** 这就是 PyTorch autograd 的核心机制。---### 14.4　用 PyTorch 验证手算结果纸上算完，然后用 PyTorch 验证——这是深入理解 autograd 的最佳方式。💻 **代码　PyTorch autograd 验证手工反向传播**

In [ ]:
# 环境要求：PyTorch 2.0+import torch# y = (x+2) × 3 —— 手工反向传播# 前向: x=1 → u=x+2=3 → y=u*3=9# 反向: ∂y/∂y=1, ∂y/∂u=3, ∂u/∂x=1 → ∂y/∂x=3×1=3x = torch.tensor(1.0, requires_grad=True)u = x + 2      # 加节点——本地梯度=1y = u * 3      # 乘节点——本地梯度=3y.backward()   # 触发反向传播！print(f"手工计算: y = {y.item()}, ∂y/∂x = 3")print(f"PyTorch:   y = {y.item()}, ∂y/∂x = {x.grad.item()} ✓")# 验证更复杂的：y = sin(x²) 在 x=2x2 = torch.tensor(2.0, requires_grad=True)y2 = torch.sin(x2 ** 2)y2.backward()expected = torch.cos(torch.tensor(4.0)) * 4  # cos(4)×4print(f"\nsin(x²) at x=2: PyTorch grad = {x2.grad.item():.6f}")print(f"链式法则: cos(4)×4 = {expected.item():.6f}  匹配: ✓")

> **关键洞察**：PyTorch 不需要你手动拆解 `sin(x²)` 成内层和外层——它在你写 `torch.sin(x**2)` 时自动构建了计算图，然后 `backward()` 自动遍历执行链式法则。**你写的每一行 PyTorch 代码都在隐式地建图。**🔗 **AI 连接**：PyTorch 的 `.backward()` 只能对**标量**调用（loss 是一个标量）。如果你对一个向量调用 `.backward()`，需要传一个和它同 shape 的 `gradient` 参数——这对应链式法则的"上游梯度"输入。---### 14.5　为什么 PyTorch 用动态计算图？PyTorch 之前的框架（TensorFlow 1.x）用**静态图**——先定义整张计算图，再喂数据执行。像先画好建筑施工图再动工。优点是能全局优化，缺点是每次换输入形状就要重新画图。PyTorch 用**动态图（Define-by-Run）**——每次前向传播时即时构建计算图，forward 完图就建好了，backward 完图就丢弃了。**像即兴发挥而不是照图施工。** 对于变长输入（NLP 中句子长度不一）、条件分支（if 语句选不同路径）、调试（可以用 `print` 和 `pdb` 随便打断点），动态图都更灵活。丧失的是"全局优化"——但 PyTorch 2.0 的 `torch.compile` 通过 JIT 编译把动态图转成优化后的静态图，把两者的优点结合了。---### 14.6　切断梯度：detach() 与 no_grad() —— 控制梯度流动反向传播默认从 loss 一路传到所有 `requires_grad=True` 的叶子节点。但有时你需要**阻止**梯度流动——两种工具，两种场景：| 工具 | 作用范围 | 典型场景 ||------|---------|---------|| `.detach()` | 单个张量 | 拿模型输出做可视化分析，但这个输出不应影响训练 || `torch.no_grad()` | 整个代码块 | 推理（生成文本）、评估（算准确率）、保存 checkpoint |📐 **定义**：- **`.detach()`**：创建张量的副本，与原计算图断开——新张量 `requires_grad=False`。- **`torch.no_grad()`**：上下文管理器，进入后所有操作都不构建计算图，节省显存和计算时间。💻 **代码　detach 断开单个张量 vs no_grad 关闭整块梯度追踪**

In [ ]:
import torch# ===== .detach(): 断开单个张量 =====w = torch.tensor([2.0, 3.0], requires_grad=True)x = torch.tensor([1.0, 2.0])y = (w * x).sum()        # y 在计算图中，w 的梯度会被追踪# 用 detach 拿一份 y 的值——这和训练无关，只是看看y_value = y.detach()     # y_value 不在计算图中print(f"y = {y.item():.2f}, y_value = {y_value}")print(f"y.requires_grad = {y.requires_grad}")         # True —— 在图里print(f"y_value.requires_grad = {y_value.requires_grad}")  # False —— 断开了# ===== torch.no_grad(): 整个代码块不建图 =====import timex_big = torch.randn(1000, 1000)# 有梯度追踪t0 = time.perf_counter()for _ in range(200):    y = x_big @ x_big.T; y.sum()t1 = time.perf_counter()# no_grad：不建图——推理和评估的标准做法with torch.no_grad():    t2 = time.perf_counter()    for _ in range(200):        y = x_big @ x_big.T; y.sum()    t3 = time.perf_counter()print(f"\n有梯度追踪: {t1-t0:.3f}s")print(f"no_grad:    {t3-t2:.3f}s")print(f"加速:       {(t1-t0)/(t3-t2):.1f}x  ← 省了建图 + 省了显存")# ===== 常见陷阱：detach 不修改原张量！ =====x_orig = torch.tensor([5.0], requires_grad=True)x_detached = x_orig.detach()    # 创建一个新张量，和 x_orig 无关print(f"\nx_orig.requires_grad = {x_orig.requires_grad}")      # 还是 True!print(f"x_detached.requires_grad = {x_detached.requires_grad}") # Falseprint("关键: detach() 不修改原张量——它返回一个新张量")

> **关键洞察**：`no_grad()` 下的运算不仅省显存（不存储中间结果），还省时间（不构建计算图）。推理时全程 `with torch.no_grad()`，评估时 `model.eval()` + `torch.no_grad()` 是标准组合。`.detach()` 则用于更精细的控制——比如 GAN 训练中更新生成器时需要冻结判别器的梯度，但判别器的梯度不能全局关闭。🔗 **AI 连接**：第 30 章 GPT-2 文本生成全程在 `torch.no_grad()` 下运行；第 29 章 Transformer 推理时用 KV Cache 配合 `no_grad`；第 14 章的练习中你会对比有/无 `no_grad` 的显存和速度差异。---### 14.7　Agent 中的计算图陷阱：多轮对话的显存管理现在把 `detach` 和 `no_grad` 的知识应用到一个真实的 Agent 工程问题。一个 ReAct Agent 的典型交互模式是"思考→行动→观察→思考→..."——每轮对话都在前一轮的基础上累积新的 token。如果你用 PyTorch 追踪了所有历史轮次的计算图，**显存将随对话轮数线性增长**：第 1 轮存 1 份中间结果，第 2 轮存 2 份，第 10 轮存 10 份——一个 10 轮对话可能吃掉比单轮多 10 倍的显存。**解决方案**：每轮对话结束时，对历史状态调用 `.detach()`——切断历史轮次的梯度回传，只保留当前轮的数值。这样无论对话进行多少轮，显存占用保持恒定。📐 **核心原则**：**在 Agent 多轮交互中，每一轮的输出对下一轮是"数据"而非"计算图节点"。** 历史是用来读的，不是用来反向传播的。`.detach()` 就是这个"只读"开关。💻 **代码　模拟 10 轮对话：有/无 detach 的显存对比**

In [ ]:
import torchn_rounds = 6d_model = 256  # 缩小维度便于演示，真实 LLM 用 4096+# ===== 无 detach：计算图随轮数增长 =====print("=== 无 detach（计算图滚雪球）===")state = torch.randn(1, d_model, requires_grad=True)for r in range(n_rounds):    W = torch.randn(d_model, d_model, requires_grad=True)    state = state @ W                             # state 的 grad_fn 链越来越长！    print(f"  第{r+1}轮: state.grad_fn = {str(state.grad_fn)[:50]}...")# ===== 有 detach：每轮切断历史，保持轻量 =====print("\n=== 有 detach（每轮切断历史）===")state = torch.randn(1, d_model, requires_grad=True)for r in range(n_rounds):    W = torch.randn(d_model, d_model, requires_grad=True)    new_state = state @ W    state = new_state.detach().requires_grad_(True)  # 切断！从零开始建图    print(f"  第{r+1}轮: state.grad_fn = {state.grad_fn}  ← 始终为 None（叶子节点）")print(f"\n无 detach：grad_fn 链条越来越长 → 反向传播穿越所有历史轮次 → 显存 O(n)")print(f"有 detach：grad_fn 始终为 None → 每轮独立 → 显存 O(1)")print(f"结论：多轮 Agent 必须在每轮结束时 detach()，否则显存线性增长直到 OOM")

> **关键洞察**：`state = new_state.detach()` 这行代码是 Agent 多轮对话系统中最重要的显存管理操作——它告诉 PyTorch："上一轮的输出只是这一轮的输入数据，不需要追踪从第一轮到当前轮的完整计算图"。**一个 `.detach()` 调用，显存从 O(n) 降为 O(1)。** 在真实 LLM 推理（数十 GB 显存）中，忘记 detach 会在 5-10 轮对话后直接 OOM 崩溃。🔗 **AI 连接**：第 5.5 节 RAG 检索到的文档、第 4.5 节的状态迁移、第 6.6 节的连续批处理——都需要在多轮交互中配合 `.detach()` 控制显存。第 30 章 GPT-2 推理全程 `torch.no_grad()`，第 31 章训练循环中 `.zero_grad()` 清空梯度——和 `.detach()` 一起构成 Agent 系统的"显存三件套"。---**✏️ 习题**1. （概念）画出 `y = (x×2 + 1)²` 在 x=3 时的计算图，标注每个节点的值和本地梯度。手算反向传播，给出 ∂y/∂x 的结果。2. （概念）`.detach()` 和 `torch.no_grad()` 的核心区别是什么？分别适用于什么场景？3. （代码）对习题 1 的表达式，用 PyTorch 的 `backward()` 验证手算结果。额外：在表达式中间插入一个 `detach()` 节点，观察梯度是否被截断。4. （代码）构造一个 500×500 的随机矩阵，分别在有梯度和 `no_grad()` 下循环做 500 次 `X @ X.T`，对比耗时和显存。5. （代码）模拟 10 轮 Agent 对话：每轮对状态向量做线性变换 → backward → 下一轮。对比 `.detach()` 和不断开梯度时，`requires_grad=True` 的张量数量增长趋势。验证"不 detach 时显存随轮数线性增长，detach 后保持恒定"。---> 🔗 **章末钩子**：你现在理解了 PyTorch 的内部原理——计算图建图、自动微分、梯度控制。但梯度是怎么从"不确定性"中产生的？如果模型 100% 确定答案，梯度就是零——不确定性用概率来量化。> 预览：第四部分——**概率论与统计**，第 15 章从随机变量与分布开始。用采样代替公式，理解不确定性的数学语言。